# Health facilities, buffers, and service-distance thinking

This notebook uses real-world health facility coordinates as a small teaching layer. It pairs point locations with OpenStreetMap basemaps and simple browser-safe distance calculations.

The goal is not to replace network analysis; it is to teach how quickly a point layer changes a policy map.

Relevant data source to explore next: healthsites.io for open health-facility mapping workflows.

**Reflection questions:** Which populations are invisible when you only map facilities? How would travel-time analysis differ from straight-line distance? What data-quality checks should precede operational use?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
fac = load_csv('health_facilities_training_points.csv')
# Haversine distance in kilometers: Pyodide-safe and dependency-free.
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

# Example demand point near Times Square.
demand = {'name':'Example demand point: Times Square', 'lat':40.7580, 'lon':-73.9855}
nyc = fac[fac.city.eq('New York')].copy()
nyc['km_from_demand'] = nyc.apply(lambda r: haversine(demand['lat'], demand['lon'], r.lat, r.lon), axis=1)
nyc[['name','km_from_demand']].sort_values('km_from_demand')

In [ ]:
m = folium.Map(location=[40.74, -73.95], zoom_start=11, tiles='OpenStreetMap')
folium.Marker([demand['lat'], demand['lon']], tooltip=demand['name'], icon=folium.Icon(color='red', icon='info-sign')).add_to(m)
cluster = MarkerCluster(name='Facilities').add_to(m)
for _, r in nyc.iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=7, fill=True, popup=f"{r['name']}<br>{r.km_from_demand:.1f} km from demand point").add_to(cluster)
    folium.PolyLine([[demand['lat'], demand['lon']], [r.lat, r.lon]], tooltip=f"{r.km_from_demand:.1f} km").add_to(m)
for km in [2,5,10]:
    folium.Circle([demand['lat'], demand['lon']], radius=km*1000, fill=False, tooltip=f'{km} km straight-line ring').add_to(m)
add_standard_controls(m)
m